EXPLORACIÓN DEL DATASET NBA

QUÉ ES: Este notebook es para explorar el dataset, me va a ayudar a leer los ficheros originales y generarme un informe de perfil. Para cada uno de los cuatro ficheros va a recoger su tamaño, las filas duplicadas y por columna, el tipo de dato, el porcentaje de nulos y los valores distintos. 

He tomado este paso por tres motivos:

    1. El fichero principal supera el millon de filas, lo que hace imposible revisarlo a mano o en una hoja de cálculo sin perder datos. 

    2. Permite que cada cifra de la ficha tenga una fuente verificable. 

    3. Siguiendo las lecciones 02 y 03, documentar el estado de los datos el día de la descarga y cuantificar sus problemas de calidad desde el principio sirve como punto de partida para la fase de limpieza y para detectar cambios si la fuente se actualiza. 

DE DÓNDE SALEN LOS DATOS: NBA Dataset: Box Scores and STats (1947 - Today), de EUIN A.Moore, en Kaggle, licencia CC0. 

ACLARACIÓN: el notebook solo lee los ficheros originales y no los modifica. 


In [3]:
##Preparacion
from pathlib import Path #trae la herramienta para manejar rutas de carpetas
import pandas as pd #trae pandas

RAW =Path("..")/"data"/"raw" #guarda donde estan los CSV 
                             #.. es "sube una carpeta", desde notebooks/a project/.
pd.set_option("display.max_columns", None)

In [4]:
#cargar Players.csv y mostrar sus primeras filas

players = pd.read_csv(RAW/"Players.csv")
players.head()

,personId,firstName,lastName,birthDate,school,country,heightInches,bodyWeightLbs,jersey,guard,forward,center,dleagueFlag,nbaFlag,gamesPlayedFlag,draftYear,draftRound,draftNumber,fromYear,toYear
0,196294534,Alberto,Abalde,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,76001,Alaa,Abdelnaby,1968-06-24,Duke,USA,82.0,240.0,30,0,1,0,0.0,1.0,1.0,1990.0,1.0,25.0,1990.0,1994.0
2,76002,Zaid,Abdul-Aziz,1946-04-07,Iowa State,USA,81.0,235.0,54,0,0,1,0.0,1.0,1.0,1968.0,1.0,5.0,1968.0,1977.0
3,76003,Kareem,Abdul-Jabbar,1947-04-16,UCLA,USA,86.0,225.0,33,0,0,1,0.0,1.0,1.0,1969.0,1.0,1.0,1969.0,1988.0
4,51,Mahmoud,Abdul-Rauf,1969-03-09,Louisiana State,USA,73.0,162.0,1,1,0,0,0.0,1.0,1.0,1990.0,1.0,3.0,1990.0,2000.0


## ¿QUÉ CONTIENE LA TABLA?
```
personID: identificador
Nombre y apellidos (campos separados)
Fecha de nacimiento
Universidad
País
Altura (pero en nomenclatura rara)
Peso
Numero camiseta
Posición
dleagFlag y nbaFlag
Partidos jugados en Flag
Año de Draft
Draft round
Draft number
De que año a que año
```

El primer hallazco de calidad: ALBERTO(0) no tiene fecha de nacimiento con la cual quiero calcular la edad para asociar que cuanto mayor es el jugador hay posibilidad de más fatiga.

Otro dato que me llama la atención es que dleagueFlag y nbaFlag sus datos estan en decimal .0, 0.0 y 1.0, creo que quieren decir 0 y 1... hay que corregir e indagar más adelante en ello. 

In [5]:
#cargar los otros tres csv y mostrar sus primeras filas
#cargar Games.cvs
games = pd.read_csv(RAW/"Games.csv", low_memory=False) #low_memory=False evita un warning
games.head()

,gameId,gameDateTimeEst,hometeamCity,hometeamName,hometeamId,awayteamCity,awayteamName,awayteamId,homeScore,awayScore,winner,gameType,gameSubtype,gameLabel,gameSubLabel,seriesGameNumber,attendance,arenaId,arenaName,arenaCity,arenaState,officials,gameDate
0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,90,94,1610612752,Playoffs,NaN,NBA Finals,Game 5,Game 5,18984.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, Scott Foster, Mitchell Ervin, Ty...",2026-06-13 20:30:00
1,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,107,106,1610612752,Playoffs,NaN,NBA Finals,Game 4,Game 4,19812.0,30,Madison Square Garden,New York,NY,"Courtney Kirkland, Zach Zarba, James Williams,...",2026-06-10 20:30:00
2,42500403,2026-06-08 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,111,115,1610612759,Playoffs,NaN,NBA Finals,Game 3,Game 3,19812.0,30,Madison Square Garden,New York,NY,"Marc Davis, John Goble, Curtis Blair, Nick Buc...",2026-06-08 20:30:00
3,42500402,2026-06-05 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,104,105,1610612752,Playoffs,NaN,NBA Finals,Game 2,Game 2,19014.0,1000118,Frost Bank Center,San Antonio,TX,"Tony Brothers, Josh Tiven, Mitchell Ervin, Tyl...",2026-06-05 20:30:00
4,42500401,2026-06-03 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,95,105,1610612752,Playoffs,NaN,NBA Finals,Game 1,Game 1,18835.0,1000118,Frost Bank Center,San Antonio,TX,"James Capers, JB DeRosa, Scott Foster, Sean Wr...",2026-06-03 20:30:00


## ¿Qué hay en la tabla?

```
gameId                gameDateTimeEst       hometeamCity
hometeamName          hometeamId            awayteamCity
awayteamName          awayteamId            homeScore
awayScore             winner                gameType
gameSubtype           gameLabel             gameSubLabel
seriesGameNumber      attendance            arenaId
arenaName             arenaCity             arenaState
officials             gameDate
```


In [6]:
#cargar TeamStatistics.csv
team_stats = pd.read_csv(RAW/"TeamStatistics.csv")
team_stats.head()

,gameId,gameDateTimeEst,teamCity,teamName,teamId,opponentTeamCity,opponentTeamName,opponentTeamId,home,win,teamScore,opponentScore,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,numMinutes,q1Points,q2Points,q3Points,q4Points,benchPoints,biggestLead,biggestScoringRun,leadChanges,pointsFastBreak,pointsFromTurnovers,pointsInThePaint,pointsSecondChance,timesTied,timeoutsRemaining,seasonWins,seasonLosses,coachId,gameType,gameLabel,gameSubLabel,seriesGameNumber,seed,reboundsTeam,turnoversTeam,ot1Points,ot2Points,otAllPoints,gameDate
0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,1,0.0,90,94,18.0,7.0,6.0,86.0,33.0,0.384,37.0,12.0,0.324,19.0,12.0,0.632,33.0,14.0,47.0,22.0,12.0,-4.0,240.0,23.0,19.0,30.0,18.0,32.0,16.0,9.0,5.0,13.0,21.0,38.0,17.0,5.0,0.0,1.0,4.0,NaN,Playoffs,NBA Finals,Game 5,5.0,2.0,12.0,1.0,NaN,NaN,NaN,2026-06-13 20:30:00
1,42500405,2026-06-13 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,0,1.0,94,90,14.0,4.0,8.0,87.0,31.0,0.356,37.0,12.0,0.324,28.0,20.0,0.714,35.0,13.0,48.0,21.0,10.0,4.0,240.0,13.0,24.0,28.0,29.0,9.0,4.0,10.0,5.0,13.0,18.0,30.0,14.0,5.0,0.0,4.0,1.0,NaN,Playoffs,NBA Finals,Game 5,5.0,3.0,18.0,4.0,NaN,NaN,NaN,2026-06-13 20:30:00
2,42500404,2026-06-10 20:30:00,New York,Knicks,1610612752,San Antonio,Spurs,1610612759,1,1.0,107,106,23.0,4.0,6.0,78.0,36.0,0.462,32.0,15.0,0.469,28.0,20.0,0.714,31.0,8.0,39.0,21.0,13.0,1.0,240.0,22.0,27.0,26.0,32.0,12.0,1.0,13.0,3.0,13.0,11.0,34.0,14.0,2.0,0.0,3.0,1.0,NaN,Playoffs,NBA Finals,Game 4,4.0,3.0,14.0,2.0,NaN,NaN,NaN,2026-06-10 20:30:00
3,42500404,2026-06-10 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,0,0.0,106,107,24.0,4.0,10.0,86.0,36.0,0.419,43.0,17.0,0.395,20.0,17.0,0.850,30.0,12.0,42.0,21.0,11.0,-1.0,240.0,41.0,35.0,14.0,16.0,28.0,29.0,10.0,3.0,12.0,25.0,28.0,10.0,2.0,0.0,1.0,3.0,NaN,Playoffs,NBA Finals,Game 4,4.0,2.0,8.0,1.0,NaN,NaN,NaN,2026-06-10 20:30:00
4,42500403,2026-06-08 20:30:00,San Antonio,Spurs,1610612759,New York,Knicks,1610612752,0,1.0,115,111,28.0,7.0,7.0,84.0,39.0,0.464,34.0,12.0,0.353,32.0,25.0,0.781,31.0,6.0,37.0,21.0,8.0,4.0,240.0,33.0,24.0,35.0,23.0,25.0,12.0,7.0,6.0,5.0,21.0,44.0,10.0,4.0,0.0,1.0,2.0,NaN,Playoffs,NBA Finals,Game 3,3.0,2.0,11.0,0.0,NaN,NaN,NaN,2026-06-08 20:30:00


## ¿Qué hay en la tabla?

```
gameId                    gameDateTimeEst           teamCity                  teamName
teamId                    opponentTeamCity          opponentTeamName          opponentTeamId
home                      win                       teamScore                 opponentScore
assists                   blocks                    steals                    fieldGoalsAttempted
fieldGoalsMade            fieldGoalsPercentage      threePointersAttempted    threePointersMade
threePointersPercentage   freeThrowsAttempted       freeThrowsMade            freeThrowsPercentage
reboundsDefensive         reboundsOffensive         reboundsTotal             foulsPersonal
turnovers                 plusMinusPoints           numMinutes                q1Points
q2Points                  q3Points                  q4Points                  benchPoints
biggestLead               biggestScoringRun         leadChanges               pointsFastBreak
pointsFromTurnovers       pointsInThePaint          pointsSecondChance        timesTied
timeoutsRemaining         seasonWins                seasonLosses              coachId
gameType                  gameLabel                 gameSubLabel              seriesGameNumber
seed                      reboundsTeam              turnoversTeam             ot1Points
ot2Points                 otAllPoints               gameDate
```

Tengo **seasonWins** y **seasonLosses**, con ellas puedo medir el nivel del rival -> descansar contra rivales débiles. 

In [7]:
#cargar PlayerStatistics.csv
player_stats = pd.read_csv(RAW/"PlayerStatistics.csv", low_memory=False) #low_memory=False evita un warning
player_stats.head()

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,gameLabel,gameSubLabel,seriesGameNumber,win,home,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate
0,Josh,Hart,1628404.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,39.166666666666664,13.0,2.0,0.0,0.0,11.0,4.0,0.364,6.0,3.0,0.5,3.0,2.0,0.667,8.0,3.0,11.0,3.0,1.0,15.0,1.610613e+09,1.610613e+09,NaN,F,2026-06-13 20:30:00
1,OG,Anunoby,1628384.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,32.78333333333333,11.0,0.0,1.0,3.0,11.0,3.0,0.273,5.0,1.0,0.2,6.0,4.0,0.667,8.0,0.0,8.0,5.0,1.0,2.0,1.610613e+09,1.610613e+09,NaN,F,2026-06-13 20:30:00
2,Jeremy,Sochan,1631110.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,0.15,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,1.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-06-13 20:30:00
3,Jordan,McLaughlin,1629162.0,42500405,2026-06-13 20:30:00,San Antonio,Spurs,New York,Knicks,Playoffs,NBA Finals,Game 5,5,0.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.0,0.0,0.0,0.0,1.610613e+09,1.610613e+09,DNP - Coach's Decision,NaN,2026-06-13 20:30:00
4,Jordan,Clarkson,203903.0,42500405,2026-06-13 20:30:00,New York,Knicks,San Antonio,Spurs,Playoffs,NBA Finals,Game 5,5,1.0,0.0,6.116666666666666,2.0,0.0,0.0,0.0,5.0,1.0,0.200,2.0,0.0,0.0,0.0,0.0,0.000,0.0,1.0,1.0,0.0,0.0,3.0,1.610613e+09,1.610613e+09,NaN,NaN,2026-06-13 20:30:00


## ¿Qué hay en la tabla?

```
firstName                 lastName                  personId
gameId                    gameDateTimeEst           playerteamCity
playerteamName            opponentteamCity          opponentteamName
gameType                  gameLabel                 gameSubLabel
seriesGameNumber          win                       home
numMinutes                points                    assists
blocks                    steals                    fieldGoalsAttempted
fieldGoalsMade            fieldGoalsPercentage      threePointersAttempted
threePointersMade         threePointersPercentage   freeThrowsAttempted
freeThrowsMade            freeThrowsPercentage      reboundsDefensive
reboundsOffensive         reboundsTotal             foulsPersonal
turnovers                 plusMinusPoints           playerteamId
opponentteamId            comment                   startingPosition
gameDate
```

Esta tabla tiene mucha información que va a ser esencial para mi sección de carga:

- **Carga** (minutos jugados) → `numMinutes`
- **Fecha** (calcular descansos y back-to-backs) → `gameDateTimeEst` / `gameDate`
- **Rendimiento** → `points`, `plusMinusPoints`, los tiros
- **Contexto** → `home`, `win`, `opponentteamName`, `gameType`
- **Jugador** → `personId`

Por otro lado, **comment** creo que va a ser oro par mi proyecto: En la 4ª fila pone “DNP - Coach’s Decision”. DNP significa Did Not Play: el jugador no jugó porque el entrenador lo decidió.Si esta columna recogiera otros motivos, como lesión, tendría una aproximación a lesiones y descansos. 



## TAMAÑO, TIPOS DE DATOS Y NULOS

En este bloque mido cada tabla:
- cuantas filas y columnas tiene, 
- qué tipo de dato hay en cada columna, 
- cuántos valores faltan. 


### Tamaño de cada tabla


In [9]:
print("players:", players.shape) 
print("games:", games.shape) #con shape se puede ver cuantas filas y columnas tiene el dataframe
print("team_stats:", team_stats.shape)
print("player_stats:", player_stats.shape)
#resultado tapla_name : (filas, columnas)

players: (6692, 20)
games: (73279, 23)
team_stats: (146560, 59)
player_stats: (1669922, 40)


### Tipos de dato

Aqui voy a revisar qué tipo ha asignado pandas a cada columna de las dos tablas que más voy a usar: `player_stats` y `players`.

In [10]:
player_stats.dtypes

firstName                      str
lastName                       str
personId                   float64
gameId                       int64
gameDateTimeEst                str
playerteamCity                 str
playerteamName                 str
opponentteamCity               str
opponentteamName               str
gameType                       str
gameLabel                      str
gameSubLabel                   str
seriesGameNumber               str
win                        float64
home                       float64
numMinutes                     str
points                     float64
assists                    float64
blocks                     float64
steals                     float64
fieldGoalsAttempted        float64
fieldGoalsMade             float64
fieldGoalsPercentage       float64
threePointersAttempted     float64
threePointersMade          float64
threePointersPercentage    float64
freeThrowsAttempted        float64
freeThrowsMade             float64
freeThrowsPercentage

**Observaciones**: numMinutes es texto, y es una de mis variables principales sobre las que quiero trabajar. Algo pasa... voy a sarcar 20 filas al azar a ver que es lo que esta pasando.

In [11]:
player_stats["numMinutes"].sample(20)

1416174    12.0
1369094    27.0
1575544    41.0
663967     13.0
1336325     2.0
145369     31.0
529241      NaN
1189458    43.0
406595     13.0
1185190    40.0
177251     31.0
119553      6.0
443560     27.0
968911     48.0
1657091     0.0
686586      5.0
1595529     0.0
768864      6.0
873856      3.0
162505      NaN
Name: numMinutes, dtype: str

La muestra parece normal, casi todo son números y dos NaN. Aun así me sigue molestando que dtype sea str. 

Lo que puedo ver **ahora mismo**:
- Los minutos estan redondeados a enteros, lo que me quita precisión y debo comentarlo cuando llegue a mis conclusiones. 
- Hay 0.0 y NaN que no es lo mismo. 0.0 significaría que el jugados estaba en el partido pero no ha jugado nada y NaN es que el dato falta. 

Debería encontrar los valores raros que hacen que los minutos se declaren como str. 


In [12]:
# Intento convertir numMinutes a número; lo que no se pueda convertir se queda vacío (NaN)
minutos_num = pd.to_numeric(player_stats["numMinutes"], errors="coerce")

# Valores que NO estaban vacíos en el original pero SÍ quedan vacíos al convertir: esos son los raros
raros = player_stats.loc[minutos_num.isna() & player_stats["numMinutes"].notna(), "numMinutes"]

print("Filas con valores no numéricos:", len(raros))
raros.value_counts().head(20)

Filas con valores no numéricos: 2097


numMinutes
12:00    28
22:29     7
23:46     6
15:51     6
16:08     6
14:53     6
7:04      6
15:43     6
16:22     5
8:27      5
21:18     5
21:15     5
1:54      5
17:16     5
16:07     5
17:11     5
20:47     5
16:42     5
16:53     5
17:34     5
Name: count, dtype: int64

Ya tengo el culpable de que me salga el tipo str: tiene 2097 filas en formato minutos:segundos y el resto utiliza un número decimal. 

Al saber la causa, hay que resolverlo bien. Plantearme si descartar todas las filas ya que de 1,6 millones de filas son solo 0,1 % aproximadamente o si deberia convertir el 22:34 en 22,34. 



In [13]:
players.dtypes

personId             int64
firstName              str
lastName               str
birthDate              str
school                 str
country                str
heightInches       float64
bodyWeightLbs      float64
jersey                 str
guard                int64
forward              int64
center               int64
dleagueFlag        float64
nbaFlag            float64
gamesPlayedFlag    float64
draftYear          float64
draftRound         float64
draftNumber        float64
fromYear           float64
toYear             float64
dtype: object

Observaciones: 
- 'birthDate' es una cadena de código si quiero calcular la edad tendre que convertirla. 
- las unidades de 'hightInches' y 'bodyWeightLbs' estan en pulgadas y libras tendre que cambiarlas a cm y kg ya que estoy en España. 

## Valores nulos

Voy a calcular el % de valores vacíos de cada columna,ordenado de mayor a menor. 

In [14]:
##NULOS DE PLAYER_STATS
player_stats.isna().sum() #cuenta los nulos por columna
#.isna() convierte cada celda en True= 1(vacia) o False =0 (con dato).
#.sum()suma los 1 

firstName                      234
lastName                       238
personId                        45
gameId                           0
gameDateTimeEst               3513
playerteamCity                 168
playerteamName                3513
opponentteamCity              3681
opponentteamName              3513
gameType                        50
gameLabel                  1552474
gameSubLabel               1564831
seriesGameNumber           1532465
win                           3523
home                          3513
numMinutes                  168636
points                        1687
assists                       1687
blocks                        1687
steals                        1687
fieldGoalsAttempted           1687
fieldGoalsMade                1687
fieldGoalsPercentage          1687
threePointersAttempted        1687
threePointersMade             1687
threePointersPercentage       1687
freeThrowsAttempted           1687
freeThrowsMade                1687
freeThrowsPercentage

In [15]:
#calcular el porcentaje de nulos por columna y ordenar de mayor a menor
(player_stats.isna().sum()/len(player_stats)*100).sort_values(ascending=False) 

gameSubLabel               93.706832
gameLabel                  92.966857
seriesGameNumber           91.768657
comment                    91.005628
startingPosition           65.700134
numMinutes                 10.098436
opponentteamId              6.545156
playerteamId                6.545156
opponentteamCity            0.220429
win                         0.210968
home                        0.210369
opponentteamName            0.210369
gameDateTimeEst             0.210369
playerteamName              0.210369
gameDate                    0.210369
plusMinusPoints             0.157792
reboundsTotal               0.101023
threePointersAttempted      0.101023
fieldGoalsMade              0.101023
fieldGoalsPercentage        0.101023
steals                      0.101023
assists                     0.101023
fieldGoalsAttempted         0.101023
points                      0.101023
freeThrowsMade              0.101023
reboundsOffensive           0.101023
reboundsDefensive           0.101023
f

**observaciones**: 
- gameSubLabel, gameLabel y seriesGAmeNumber tiene un 92/94% de nulos pero esto se debe a que solo tienen valor en playofs. En temporada regular no se aplican. 

- comment tiene un 91% pero es porque solo se rellena cuando el jugador no juega. si esta vacío significa que si ha jugado. 

- startingPosition tiene un 65,7% pero hay que tener presente que solo tendra un valor si el jugador ha sido o fue titular.Vacío significa que salió desde el banquillo. ** que en verdad es una buena noticia para mi hipotesis porque ,e va a dar un titular/suplente, que va a influir mucho en los minutos y la carga**

- numMinutes tiene un 10,1%, es mi variable de carga por lo que tengo que investigar porque hay nulos. 

- playerteamId, opponentteamId tiene un 6,5%, falta los IDs de los equipos pero tengo los nombres, que solo hay 0,2% de nulos por lo que podría utilizar este dato como identificador. 

La pregunta importante de esto es **¿POR QUÉ FALTAN MINUTOS?**


¿puede ser que los minutos vacios sean jugadores que no jugaron? Si es así no serían datos perdidos, sino partidos en el que el jugador no participó... y eso es justo lo que necesitaría para medir los descansos. 

In [16]:
#Voy a hacer una tabla cruzada para ver si los nulos de numMinutes se concentran en ciertos partidos o jugadores.
#¿Coinciden los minutos vacíos con los jugadores que tienen comentario (no jugaron)?
pd.crosstab(                                #cuenta cuantas filas hay en cada combiancion de dos condiciones.
    player_stats["numMinutes"].isna(),      #¿estan vacios los minutos?
    player_stats["comment"].notna(),        #¿tiene comentario?
    rownames=["minutos vacíos"],
    colnames=["tiene comentario"],
)

tiene comentario,False,True
minutos vacíos,,
False,1501276,10
True,18447,150189


**Resultado:** la hipótesis se confirma.

- De las 168.636 filas sin `numMinutes`, 150.189 (89 %) tienen comentario: el jugador **no jugó** ese partido. No son datos perdidos, sino partidos no disputados, que son justo los que necesito para medir descansos.

- Solo 18.447 filas (1,1 % del total) no tienen ni minutos ni comentario. Son los **nulos reales**. Comprobaré en el bloque de fechas si se concentran en temporadas antiguas.

- 10 filas tienen minutos y comentario a la vez; son casos aislados.

In [17]:
# Número de nulos por columna
players.isna().sum()

personId              0
firstName             1
lastName              2
birthDate          1139
school             1415
country            1399
heightInches       1585
bodyWeightLbs      1595
jersey             2868
guard                 0
forward               0
center                0
dleagueFlag        1231
nbaFlag            1231
gamesPlayedFlag    1231
draftYear          1136
draftRound         1298
draftNumber        1357
fromYear           1337
toYear             1922
dtype: int64

In [18]:
# Porcentaje de nulos por columna, ordenado de mayor a menor
(players.isna().sum() / len(players) * 100).sort_values(ascending=False)

jersey             42.857143
toYear             28.720861
bodyWeightLbs      23.834429
heightInches       23.684997
school             21.144650
country            20.905559
draftNumber        20.277944
fromYear           19.979079
draftRound         19.396294
nbaFlag            18.395099
dleagueFlag        18.395099
gamesPlayedFlag    18.395099
birthDate          17.020323
draftYear          16.975493
lastName            0.029886
firstName           0.014943
personId            0.000000
center              0.000000
guard               0.000000
forward             0.000000
dtype: float64

Hay muchas columnas con un % similar...¿Puede que haya varios jugadores que no tengan esa información como Alberto del principio que vimos que no tenia birthdate ni muchas otras variables con datos? Lo importante es que tengan fecha de nacimiento. **investigar, queda pendiente**

Tambien puedo ver varios nulos que probablemente tengan sentido como draftyear, draftround,draftnumber, porque puede que sea que son jugadores no elegidos en el draft. 

Tambien toYear puede ser que son jugadores en activo que aun o tienen año de retirada. 

**Positivos**: personID y las posiciones tienen 0 nulos.

In [19]:
#cuenta cuantas veces aparece cada valor en la columna comment
player_stats["comment"].value_counts()

comment
DNP - Coach's Decision                     120262
DND - Injury/Illness                         4397
DNP - Injury/Illness                         1747
DND - Sprained Left Ankle                     414
DND - Sprained Right Ankle                    339
                                            ...  
DNP - 1 GAME NBA SUSPENSION                     1
DNT - Fractured Left Thumb                      1
DND - Sprain of left ankle - day to day         1
DNP - NWT; sore left knee                       1
DNP - DND; strained left hamstring              1
Name: count, Length: 7891, dtype: int64

He encontrado oro!! Al principio pensaba que no tendría datos de lesiones y ahora puedo ver que si los tiene y no los tengo que deducir. 

### Cifras a destacar
- DNP - Coach's Decision -> 120262: es el motivo más frecuente con diferencia. 

- DND - Injury/Illness (4.397) + DNP - Injury/Illness (1.747): lesión o enfermedad genérica. Y además lesiones concretas: tobillo izquierdo (414), tobillo derecho (339)…

Tambien hay un **posible problema de calidad** debido a que hay 7891 categorias diferentes = mucho trabajo de limpieza. 


In [20]:
#comprobacio de cuantos casos hay de cada tipo de comentario
player_stats["comment"].str[0:10].value_counts()

comment
DNP - Coac    120568
DND - Inju      4469
DNP - Inju      1754
DND - Spra      1725
DND - Sore      1722
               ...  
DNP - Dehy         1
"DNP - Inj         1
NWT - Rota         1
"DNP -  In         1
DNP - 1 GA         1
Name: count, Length: 1603, dtype: int64

Veo que se han escrito injury de tres maneras diferentes inju, in. inj.... esto habrá que corregirlo. 

In [21]:
# Cuenta los comentarios que contienen la palabra "rest" (sin distinguir mayúsculas)
player_stats["comment"].str.contains("rest", case=False).sum()

np.int64(418)

418 me dice que **la NBA casi nunca registra el descanso como motivo propio** Porque es un porcentaje muy bajo sobre los 150.000 comentarios que existen. Seguramente se enmascare dentro de Coach's Decision.

In [22]:
#voy a ver que son exavtamente esos 418
player_stats.loc[player_stats["comment"].str.contains("rest", case=False, na=False), "comment"].value_counts().head(10)

comment
DND - Rest                              267
NWT - Rest                               74
DNP - Rest                               34
DND - rest                               14
DND - Rest.                               7
DND - REST                                5
NWT - rest                                2
DND - Right Knee Rest                     2
DND - Left Great Toe Soreness/Rest        1
DND - Left great toe soreness / Rest      1
Name: count, dtype: int64

El descanso explicito existe pero es raro y puedo observar el mismo problema de escritura que antes. 

Quiero comprobar si es que han empezado a mirar mas los descansos recientemente porque la NBA empezó a vigilar los rest days de las estrellas alrededor del 2017. 

In [23]:
player_stats.loc[player_stats["comment"].str.contains("rest", case=False, na=False), "gameDate"].str[:4].value_counts().sort_index()

gameDate
2004     1
2007     1
2009     8
2010     4
2011     3
2012    14
2013     7
2014    31
2015    42
2016    86
2017    56
2018    25
2019    14
2020     5
2021    38
2023     7
2024     7
2025    48
2026    21
Name: count, dtype: int64

No es lo que me esperaba, veo un pico en el 2016 (86 casos). Se que la NBA en una temporada empezó a poner sanciones por dejar descanasar a estrellas sanas pero no lo recuerdo muy bien, tendría que ivestigarlo. Pero si fuera asi puede que fuera en el 2017 y de alli los equipos enmascaran los rest llamandolos de otra manera. 

## Cobertura de fechas y duplicados

Compruebo el periodo que cubren los datos, cuántos registros hay en las temporadas recientes que usaré, si los problemas detectados afectan a esas temporadas y si hay registros duplicados.

In [24]:
#Para mirar las fechas hay que convertirlas ya que datetime esta en tipo str
#convierto gameDate a fecha en una variable nueva para que no se me modifique la original
fechas = pd.to_datetime(player_stats["gameDate"], errors="coerce") #errors="coerce" convierte a NaT los que no se puedan convertir

print("Primer partido:", fechas.min())
print("Último partido:", fechas.max())

Primer partido: 1946-11-26 19:00:00
Último partido: 2026-06-13 20:30:00


Con esta comprobación veo el periodo cubierto que es del 1946-11-26 al 2026-06-13.

In [25]:
# Filas desde la temporada 2015-16 (partidos desde octubre de 2015)
print("Filas desde 2015:", (fechas >= "2015-10-01").sum())
#fechas >= "2015-10-01 marca True en cada fila posterior a esa fecha.

Filas desde 2015: 396530


Este va a ser mi periodo del trabajo. He elegido octubre porque las temporadas de la NBA empiezan en octubre. 

In [26]:
# Filas sin minutos y sin comentario (nulos reales),
# contadas por año
nulos_reales = player_stats["numMinutes"].isna() & player_stats["comment"].isna()
fechas[nulos_reales].dt.year.value_counts().sort_index()

gameDate
1996.0      26
1997.0       1
1998.0      11
1999.0       1
2000.0    1484
2001.0     653
2002.0       2
2003.0       4
2004.0       5
2005.0       5
2006.0       7
2007.0       6
2008.0      14
2009.0       5
2010.0       1
2013.0       1
2014.0    1048
2015.0     900
2016.0     829
2017.0     696
2018.0    1542
2019.0     691
2020.0     990
2021.0    3017
2022.0    4088
2023.0     633
2024.0     649
2025.0     669
2026.0       1
Name: count, dtype: int64

**¿De que años son los nulos reales de minutos?**

Esperaba que estuvieran en años antiguos pero no ha sido así. Donde hay menos informacion es en los años que quiero trabajar. Desde 2014 hay entre 600 y 1500 filas al año, con un pico en 2021 que tal vez puedo suponer que se debe al COVID. **necesito indagar más**

In [27]:
# 5 ejemplos de nulos reales en mis temporadas, para ver qué tienen en las demás columnas
player_stats.loc[nulos_reales & (fechas >= "2015-10-01")].sample(5)

,firstName,lastName,personId,gameId,gameDateTimeEst,playerteamCity,playerteamName,opponentteamCity,opponentteamName,gameType,gameLabel,gameSubLabel,seriesGameNumber,win,home,numMinutes,points,assists,blocks,steals,fieldGoalsAttempted,fieldGoalsMade,fieldGoalsPercentage,threePointersAttempted,threePointersMade,threePointersPercentage,freeThrowsAttempted,freeThrowsMade,freeThrowsPercentage,reboundsDefensive,reboundsOffensive,reboundsTotal,foulsPersonal,turnovers,plusMinusPoints,playerteamId,opponentteamId,comment,startingPosition,gameDate
286244,Brandon,McCoy,1629000.0,11800014,2018-10-01 19:30:00,New Orleans,Pelicans,Atlanta,Hawks,Preseason,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018-10-01 19:30:00
164419,Xavier,Tillman,1630214.0,22100728,2022-01-26 20:30:00,Memphis,Grizzlies,San Antonio,Spurs,Regular Season,NaN,NaN,NaN,1.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2022-01-26 20:30:00
184948,Cassius,Stanley,1630199.0,12100015,2021-10-06 19:00:00,Detroit,Pistons,San Antonio,Spurs,Preseason,NaN,NaN,NaN,1.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2021-10-06 19:00:00
184547,Jrue,Holiday,201950.0,12100026,2021-10-08 19:30:00,Milwaukee,Bucks,Brooklyn,Nets,Preseason,NaN,NaN,NaN,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2021-10-08 19:30:00
162197,James,Harden,201935.0,22100817,2022-02-08 19:30:00,Brooklyn,Nets,Boston,Celtics,Regular Season,NaN,NaN,NaN,0.0,1.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,2022-02-08 19:30:00


He visto tres cosas:

- Todas las estadísticas están vacias. Por lo tanto puedo interpretar que el jugador no jugó, pero nadie anotó el motivo. Lo que serían 0 minutos jugados por motivo desconocido. 

- Dos de los cinco son de pretemporada, el dataset mezcla tipos de partido (gameType) y la pretemporada no cuenta igual ya que son partidos de preparación. 

- Los IDs de equipo están vacios en estas filas aunque el nombre de equipo si esta... tendré que identificar por el campo de nombre. 

In [28]:
# Comprobacion de que clase de partidos hay
player_stats["gameType"].value_counts()

gameType
Regular Season        1502661
Playoffs               102249
Preseason               57118
Pre Season               3513
NBA Emirates Cup         1759
All-Star Game            1511
Play-in Tournament        998
Emirates NBA Cup           30
NBA Cup                    28
in-season-knockout          5
Name: count, dtype: int64

Mezcla tipos de partidos y el preseason aparece escrito diferente. 

In [29]:
# Nulos reales en mis temporadas, por tipo de partido
player_stats.loc[nulos_reales & (fechas >= "2015-10-01"), "gameType"].value_counts()

gameType
Preseason             8733
Regular Season        5452
Playoffs               481
Play-in Tournament      36
NBA Cup                  2
Name: count, dtype: int64

Mas de la mitad de nulos son de pretemporada, que de todos modos iba a excluir. Me quedo con unas 6000 filas en partidos oficiales que por lo que he visto estan todas las estadisticas a 0, podría poner una regla clara para la limpieza como **"sin minutos + sin comentario + todas estadisticas a 0 se conclucye que no jugo por motivo no registrado"**



### Fecha de nacimiento en mis temporadas

Compruebo qué porcentaje de los jugadores que aparecen desde la temporada 2015-16 no tiene fecha de nacimiento, porque la necesito para calcular la edad.

In [30]:
# Lista de jugadores distintos que aparecen en partidos desde la temporada 2015-16
ids_recientes = player_stats.loc[fechas >= "2015-10-01", "personId"].unique()

# Me quedo en players solo con esos jugadores
recientes = players[players["personId"].isin(ids_recientes)]

print("Jugadores desde 2015-16:", len(recientes))
print("% sin birthDate:", recientes["birthDate"].isna().mean() * 100)

Jugadores desde 2015-16: 1993
% sin birthDate: 10.436527847466131


Al 10.4% de jugadores les falta la fecha de nacimiento. **Pero lo que de verdad importa es cuanto juegan los jugadores.** Si son jugadores con pocos partidos, apenas afectaría al analisis de carga, si fueran titulares con muchos minutos, sería un problema. Así que mejor comprobarlo:

In [31]:
# Jugadores recientes sin fecha de nacimiento
ids_sin_fecha = recientes.loc[recientes["birthDate"].isna(), "personId"]

# Qué % de las filas de mi ventana corresponden a esos jugadores
ventana = player_stats[fechas >= "2015-10-01"]
print("% de filas de jugadores sin birthDate:", ventana["personId"].isin(ids_sin_fecha).mean() * 100)

% de filas de jugadores sin birthDate: 0.21385519380626938


Los jugadores sin fecha son el 10% de los jugadores, pero aparecen en solo el 0,2% de las filas, es decir son jugadores que casi no juegan. Por lo tanto para mi hipotesis de la edad seria un impacto bajo. Creo que podría excluirlos sin perder información. 

### Duplicados

Compruebo si algún jugador aparece más de una vez en el mismo partido. Cada fila debería ser un jugador en un partido.

In [32]:
# Filas repetidas para el mismo jugador en el mismo partido
player_stats.duplicated(subset=["personId", "gameId"]).sum()

np.int64(0)

Hay 0 duplicados


### Conclusiones 

- **Periodo cubierto:** del 26/11/1946 al 13/06/2026. Mi ventana de trabajo empieza en la temporada 2015-16 (partidos desde el 01/10/2015): **396.530 filas**, suficientes para analizar y modelar.

- **Grano:** cada fila es un jugador en un partido. No hay duplicados (0 parejas `personId` + `gameId` repetidas).

- **Tipos de partido:** analizaré solo partidos oficiales (temporada regular, playoffs, play-in y Copa NBA). Excluyo pretemporada y All-Star. `gameType` tiene el mismo tipo escrito de varias formas y habrá que unificarlo.

- **Nulos reales de minutos:** no son antiguos, están en mi ventana (≈ 14.700 filas, 3,7 %). El 59 % son de pretemporada y desaparecen con el filtro. Los ~6.000 restantes tienen todas las estadísticas a 0: el jugador no jugó y no se registró el motivo.

- **Fecha de nacimiento:** al 10,4 % de los 1.993 jugadores desde 2015-16 les falta, pero solo representan el 0,2 % de las filas. Son jugadores con muy pocos partidos, así que el impacto en el análisis de la edad es mínimo.